# Sentiment Analysis using NLP Pipeline

End-to-End Implementation

In [2]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\asus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Load Dataset

In [14]:
df = pd.read_csv('imdb_master.csv', encoding='latin-1')
df = df[df['label'] != 'unsup']
df = df[['review', 'label']]
df.rename(columns={'review': 'text'}, inplace=True)

print(df.head())
df

                                                text label
0  Once again Mr. Costner has dragged out a movie...   neg
1  This is an example of why the majority of acti...   neg
2  First of all I hate those moronic rappers, who...   neg
3  Not even the Beatles could write songs everyon...   neg
4  Brass pictures (movies is not a fitting word f...   neg


,text,label
0,Once again Mr. Costner has dragged out a movie...,neg
1,This is an example of why the majority of acti...,neg
2,"First of all I hate those moronic rappers, who...",neg
3,Not even the Beatles could write songs everyon...,neg
4,Brass pictures (movies is not a fitting word f...,neg
...,...,...
49995,"Seeing as the vote average was pretty low, and...",pos
49996,"The plot had some wretched, unbelievable twist...",pos
49997,I am amazed at how this movie(and most others ...,pos
49998,A Christmas Together actually came before my t...,pos


## Preprocessing

In [6]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [stemmer.stem(w) for w in words]
    return " ".join(words)

df['clean_text'] = df['text'].apply(preprocess_text)

## Train Test Split

In [7]:
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## BoW

In [8]:
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

## TF-IDF

In [9]:
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

## Models

In [10]:
def evaluate(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, preds))
    print("Precision:", precision_score(y_test, preds, average='weighted'))
    print("Recall:", recall_score(y_test, preds, average='weighted'))
    print("F1 Score:", f1_score(y_test, preds, average='weighted'))
    print(classification_report(y_test, preds))

In [11]:
print("Logistic Regression (BoW)")
evaluate(LogisticRegression(max_iter=200), X_train_bow, X_test_bow, y_train, y_test)

Logistic Regression (BoW)


C:\Users\asus\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy: 0.888
Precision: 0.8881540385151491
Recall: 0.888
F1 Score: 0.8880018547260093
              precision    recall  f1-score   support

         neg       0.90      0.88      0.89      5055
         pos       0.88      0.90      0.89      4945

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [12]:
print("Naive Bayes (TF-IDF)")
evaluate(MultinomialNB(), X_train_tfidf, X_test_tfidf, y_train, y_test)

Naive Bayes (TF-IDF)
Accuracy: 0.8637
Precision: 0.8639951863897586
Recall: 0.8637
F1 Score: 0.86364511628048
              precision    recall  f1-score   support

         neg       0.85      0.88      0.87      5055
         pos       0.87      0.85      0.86      4945

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000



In [13]:
print("Decision Tree (BoW)")
evaluate(DecisionTreeClassifier(), X_train_bow, X_test_bow, y_train, y_test)

Decision Tree (BoW)
Accuracy: 0.7306
Precision: 0.7307023500217265
Recall: 0.7306
F1 Score: 0.7306067242928568
              precision    recall  f1-score   support

         neg       0.74      0.73      0.73      5055
         pos       0.72      0.74      0.73      4945

    accuracy                           0.73     10000
   macro avg       0.73      0.73      0.73     10000
weighted avg       0.73      0.73      0.73     10000



## Insights
- TF-IDF > BoW in most cases
- Logistic Regression performs best
- Naive Bayes is fast
- Decision Tree may overfit

## Pipeline
Raw Text → Preprocessing → Vectorization → Model → Evaluation
